In [1]:
# pull the GCP credentials securely from databricks secrets
gcp_client_email = dbutils.secrets.get(scope="candy_rush_scope", key="client_email")
gcp_project_id = dbutils.secrets.get(scope="candy_rush_scope", key="project_id")
gcp_private_key = dbutils.secrets.get(scope="candy_rush_scope", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="candy_rush_scope", key="private_key_id")

# Set them with sparkContext
sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

# gcp credentials
BUCKET = "gs://candy-rush-514202-openfoodfacts"
OFF_PATH = BUCKET + "/products.csv.gz"  
WHR_PATH = BUCKET + "/happiness.csv"

NameError: name 'dbutils' is not defined

In [2]:
import unicodedata

# set country data that needs to be fixed
FIXES = {
    "turkiye": "turkey",
    "czechia": "czech-republic",
    "taiwan-province-of-china": "taiwan",
    "hong-kong-sar-of-china": "hong-kong",
    "russian-federation": "russia",
    "republic-of-korea": "south-korea",
    "viet-nam": "vietnam",
    "cote-divoire": "cote-d-ivoire",
    "dr-congo": "democratic-republic-of-the-congo",
    "congo": "republic-of-the-congo",
    "lao-pdr": "laos",
    "republic-of-moldova": "moldova",
    "state-of-palestine": "palestine",
    "swaziland": "eswatini", 
    "gambia": "the-gambia",
}

# clean all country names and data to be matcheable 
def clean_country(name):
    # SPlit the accented letters into letter + accent, then drop if not plain ascii
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    # remove spaces and make lowercase
    name = name.strip().lower()
    #replace spaces and puncuation with dashes
    for ch in [" ", "'", "(", ")", ","]:
        name = name.replace(ch, "-")
        #Split on the dashes then drop spaces
    name = "-".join(part for part in name.split("-") if part)
    # if name is in FIXES swap open food spelling, otherwise dont
    return FIXES.get(name,name)
# turn text into a number

def to_float(x):
    try:
        return float(x)
    except ValueError:
        return None
    

In [ ]:
KEEP = ["code", "product_name", "categories_tags", "food_groups_tags", "countries_tags",
        "ingredients_tags", "sugars_100g", "additives_n", "additives_tags", "nova_group"]

off_file = sc.textFile(OFF_PATH)
header = off_file.first().lstrip("\ufeff")
cols = header.split("\t")
idx = [cols.index(c) for c in KEEP]   # raises a ValueError for any missing column
n = len(cols)

# parse each line in candy data
def parse(line):
    p = line.split("\t")
    return [p[i] for i in idx] if len(p) == n else None

# candy product rows 
off_rows = (off_file
            .filter(lambda l: l.lstrip("\ufeff") != header)
            .map(parse)
            .filter(lambda r: r is not None))

pos = {c: i for i, c in enumerate(KEEP)}   # use pos["sugars_100g"] etc. on any trimmed rows from above

NameError: name 'sc' is not defined

In [4]:
# RDD1 candy by country
# Tags that count as candy
CANDY_TAGS = set([
    "en:candies", "en:confectioneries", "en:sweets", "en:chocolate-candies",
    "en:gummies", "en:lollipops", "en:chewing-gum", "en:sugary-snacks",
])

#Retrun true if product is actually candy
def candy_matches(r):
    tags = r[pos["categories_tags"]].split(",") + r[pos["food_groups_tags"]].split(",")
    return [t for t in tags if t in CANDY_TAGS]

def is_candy(r):
    return len(candy_matches(r)) > 0

candy = off_rows.filter(is_candy).cache()   # cache so the checks below are fast

def explode_countries(r):
    product = (r[pos["code"]],
               r[pos["product_name"]].replace(",", " "),
               to_float(r[pos["sugars_100g"]]),
               r[pos["nova_group"]],
               r[pos["additives_n"]],
               r[pos["ingredients_tags"]].replace(",", "|"),
               r[pos["additives_tags"]].replace(",", "|"))
    for tag in r[pos["countries_tags"]].split(","):
        if tag.startswith("en:"):
            yield (clean_country(tag[3:]), product)

candy_by_country = candy.flatMap(explode_countries).cache()

print("(country, product) rows:", candy_by_country.count())
candy_by_country.take(5)

NameError: name 'off_rows' is not defined

In [5]:
#RDD 2 avgerage happiness by country
import csv

whr_raw = sc.textFile(WHR_PATH)
whr_header = whr_raw.first()
whr_cols = {name: i for i, name in enumerate(next(csv.reader([whr_header])))}

C_COUNTRY = whr_cols["Country name"]
C_SCORE = whr_cols["Life evaluation (3-year average)"]

happiness_by_country = (whr_raw
                        .filter(lambda line: line!=whr_header)
                        .map(lambda line: next(csv.reader([line])))
                        .filter(lambda r: r[C_COUNTRY] != "" and to_float(r[C_SCORE]) is not None)
                        .map(lambda r: (clean_country(r[C_COUNTRY]), (to_float(r[C_SCORE]),1)))
                        .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
                        .mapValues(lambda s: round(s[0]/s[1], 3))
                        .cache()
                        )

NameError: name 'sc' is not defined

In [0]:
# check matched countries

off_countries = set(candy_by_country.keys().distinct().collect())
whr_countries = set(happiness_by_country.keys().collect())

print("Matched: ", len(off_countries & whr_countries))
print("No Match: ", sorted(whr_countries - off_countries))

candy_by_country.join(happiness_by_country).take(3)


158 of 167 countries matched, about 95%

9 to be dropped['eswatini', 'lesotho', 'malawi', 'namibia', 'north-cyprus', 'somaliland-region', 'south-sudan', 'the-gambia', 'turkmenistan']

In [0]:
#Save both rdds to bucket

OUT_CANDY = BUCKET + "/preprocessed/candy_by_country"
OUT_HAPPY = BUCKET + "/preprocessed/happiness_by_country"

Path = sc._jvm.org.apache.hadoop.fs.Path
for out in [OUT_CANDY, OUT_HAPPY]:
    Path(out).getFileSystem(hconf).delete(Path(out), True)

#protect against none values
def to_line(kv):
    # write missing values (None) as empty text instead of the word "None"
    return ",".join("" if x is None else str(x) for x in (kv[0],) + kv[1])

candy_by_country.map(to_line).saveAsTextFile(OUT_CANDY)

happiness_by_country \
    .map(lambda kv: f"{kv[0]}, {kv[1]}") \
    .coalesce(1) \
    .saveAsTextFile(OUT_HAPPY)

# read a few lines back to show the save worked
print(sc.textFile(OUT_CANDY).take(3))
print(sc.textFile(OUT_HAPPY).take(3))
